# Five-Property Trustworthiness Evaluation — three models

This notebook grades **three models** on **five properties** and produces one table with one row
per model.

| Model | What it is |
|---|---|
| `tfidf_retrieval` | Keyword search. TF-IDF vectors, cosine similarity against the 388 descriptions |
| `zeroshot` | PubMedBERT bi-encoder, no fine-tuning |
| `finetuned` | The same encoder after contrastive fine-tuning on each fold's training cases |

Every model receives the same 400 cases, the same 388 descriptions, the same folds and the same
scoring code. **One predictions file is written per model and every table is built from those
three files.**

The pass marks are fixed in `PRE_SPECIFICATION.md`, committed before this notebook was run. They
are reproduced in Step 2 and are not changed by anything this notebook produces.

### Running this on a limited GPU

Models are loaded one at a time and released before the next one starts. A checkpoint is written
after every fold. **If a model's predictions file already exists, that model is skipped**, so you
can run the three models in three separate sessions if you need to.

## Step 1 — Environment

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import importlib, subprocess, sys
if importlib.util.find_spec("sentence_transformers") is None:
    print("installing sentence-transformers ...")
    subprocess.run([sys.executable,"-m","pip","install","-q","sentence-transformers"], check=False)
print("ready")

## Step 2 — Settings and the pass marks

Everything that can change a number lives here. The five marks are copied from
`PRE_SPECIFICATION.md` and each one carries its source.

In [ ]:
import re, json, math, random, gc, glob, warnings
from datetime import datetime, timezone
from collections import Counter, defaultdict
from math import comb, log2, erfc, sqrt
import numpy as np, pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*unique classes is greater.*")
pd.set_option("display.width", 230); pd.set_option("display.max_colwidth", 46)

CFG = dict(
    ENCODER="pritamdeka/S-PubMedBert-MS-MARCO", MAX_SEQ_LEN=512,
    BATCH=32,            # training batch, committed hyperparameter, do not change
    ENCODE_BATCH=16,     # inference batch, memory only, safe to lower
    EPOCHS=2, LR=2e-5,
    K_REPORT=5, K_RETRIEVE=10, K_CONF=3,
    N_FOLDS=5, SEED_CV=42, SEED_PERT=20261102, SEED_ATTR=20261102, SEED_BOOT=20261102,
    N_BOOT=10_000, ALPHA=0.05, N_BINS=10,
    GOLD_FIELD="diagnosis",
    OUTDIR="five_property_out",
    FULL_GRADIENT=False, # True runs all 11 perturbation conditions instead of the 5 needed
)

# ---- the five pass marks, from PRE_SPECIFICATION.md -------------------------------------
MARKS = dict(
    accuracy     = dict(value=0.6200, direction="min",
                        source="Lowest of three physicians on the same 400 cases"),
    robustness   = dict(value=0.9000, direction="min",
                        source="Reliability convention for decisional measures (Nunnally & Bernstein 1994; Koo & Li 2016)"),
    calibration  = dict(value=0.0786, direction="max",
                        source="95th percentile of ECE for a perfectly calibrated system at n=400, 10 bins"),
    abstention   = dict(value=0.5000, direction="min",
                        source="Workload consequence: a tool declining more than half does not reduce intake load"),
    auditability = dict(value=0.7500, direction="min",
                        source="Reliability convention for advisory measures (Koo & Li 2016)"),
)
ACCURACY_MARK = MARKS["accuracy"]["value"]   # abstention is evaluated against this same number

MODELS = ["tfidf_retrieval", "zeroshot", "finetuned"]

# perturbation conditions. The marks only need the registered four plus the clean reference.
REGISTERED = {"remove 2 findings":"removing",
              "reword 100%":"preserving",
              "fragment complete":"preserving",
              "vague moderate":"preserving"}
ALL_CONDITIONS = ["original",
                  "remove 1 finding","remove 2 findings","remove 3 findings",
                  "reword 25%","reword 50%","reword 100%",
                  "fragment partial","fragment complete",
                  "vague moderate","vague high"]
CONDITIONS = ALL_CONDITIONS if CFG["FULL_GRADIENT"] else ["original"]+list(REGISTERED)
FAMILY = {"remove 1 finding":"Removal","remove 2 findings":"Removal","remove 3 findings":"Removal",
          "reword 25%":"Rewording","reword 50%":"Rewording","reword 100%":"Rewording",
          "fragment partial":"Fragmentation","fragment complete":"Fragmentation",
          "vague moderate":"Vagueness","vague high":"Vagueness"}

os.makedirs(CFG["OUTDIR"], exist_ok=True)
random.seed(CFG["SEED_CV"]); np.random.seed(CFG["SEED_CV"])
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print("run id:", RUN_ID)
print("conditions this run:", len(CONDITIONS), "->", ", ".join(CONDITIONS))

display(pd.DataFrame([{"property":k, "pass mark":v["value"],
                       "direction":"at least" if v["direction"]=="min" else "at most",
                       "where the number comes from":v["source"]}
                      for k,v in MARKS.items()]))

## Step 3 — Load the data

Three files. The benchmark, the 388 descriptions, and the provenance registry. The description
file is rejected if it looks label-only, which is what a wrong file looks like.

In [ ]:
IN_COLAB = "google.colab" in sys.modules
def _find(pats, pool, skip=()):
    for f in pool:
        low = os.path.basename(f).lower()
        if f in skip or "provenance" in low: continue
        if any(p in low for p in pats): return f
    return None
def locate(pool=None):
    if pool is None: pool = sorted(glob.glob("*.json")) + sorted(glob.glob("*.xlsx"))
    pool = [f for f in pool if not os.path.basename(f).startswith("~$")]
    d = _find(("descri",), pool)
    return _find(("avey","benchmark","master","vignette"), pool, skip=(d,) if d else ()), d

PATH_DATA, PATH_DESC = locate()
if PATH_DATA is None and IN_COLAB:
    print("Upload avey_benchmark_MASTER.json, candidate_descriptions.json, "
          "candidate_descriptions_provenance.json\n")
    from google.colab import files
    PATH_DATA, PATH_DESC = locate(sorted(files.upload().keys()))
if PATH_DATA is None: raise FileNotFoundError("benchmark file not found")

RAW = json.load(open(PATH_DATA, encoding="utf-8")); CASES = RAW["cases"]
DESCRIPTIONS = json.load(open(PATH_DESC, encoding="utf-8")) if PATH_DESC else {}
_p = next((f for f in sorted(glob.glob("*.json"))
           if "provenance" in os.path.basename(f).lower()), None)
PROV = json.load(open(_p, encoding="utf-8")) if _p else {}

_med = int(np.median([len(str(v).split()) for v in DESCRIPTIONS.values()])) if DESCRIPTIONS else 0
if _med < 8:
    raise ValueError(f"descriptions look label-only (median {_med} words) - wrong file picked up")

display(pd.DataFrame([
 {"what":"patient cases","file":os.path.basename(PATH_DATA),"count":len(CASES)},
 {"what":"diagnosis descriptions","file":os.path.basename(PATH_DESC),"count":len(DESCRIPTIONS)},
 {"what":"description median length","file":"-","count":f"{_med} words"},
 {"what":"provenance records","file":os.path.basename(_p) if _p else "none","count":len(PROV)},
]))

## Step 4 — Scoring and statistics

Carried over unchanged from RQ1 to RQ3, so the three models are judged by exactly the code the
earlier chapters used.

In [ ]:
def norm(s): return " ".join(str(s).lower().strip().split())
def rank_of(r,t):
    try: return r.index(t)+1
    except ValueError: return None
def hit_at_k(r,t,k):
    x = rank_of(r,t); return bool(x is not None and x<=k)

def softmax(s, tau):
    z = np.asarray(s,float)/max(tau,1e-6); z = z - z.max(axis=-1, keepdims=True)
    e = np.exp(z); return e/e.sum(axis=-1, keepdims=True)

def topk_conf(S, tau, k=None):
    k = k or CFG["K_CONF"]
    return np.sort(softmax(S,tau), axis=1)[:,-k:].sum(axis=1)

def fit_temperature(S, gold_idx, lo=0.001, hi=10.0, iters=120):
    """Golden-section search on tau, minimising NLL of the true class on TRAINING cases."""
    S = np.asarray(S,float); gold_idx = np.asarray(gold_idx,int)
    ok = gold_idx>=0; S = S[ok]; gold_idx = gold_idx[ok]
    if len(S)==0: return 1.0
    def nll(tau):
        P = softmax(S,tau)
        return float(-np.mean(np.log(np.clip(P[np.arange(len(P)),gold_idx],1e-12,None))))
    gr=(sqrt(5)-1)/2; a,b=lo,hi; c,d=b-gr*(b-a), a+gr*(b-a)
    for _ in range(iters):
        if nll(c)<nll(d): b=d
        else: a=c
        c,d = b-gr*(b-a), a+gr*(b-a)
        if abs(b-a)<1e-5: break
    tau=float((a+b)/2)
    if tau<=lo*1.01 or tau>=hi*0.99:
        print(f"   WARNING: fitted tau = {tau:.5f} sits on the search boundary.")
    return tau

def ece_bins(conf, correct, n_bins=None):
    n_bins = n_bins or CFG["N_BINS"]
    conf=np.asarray(conf,float); y=np.asarray(correct,float)
    o=np.argsort(conf); conf,y = conf[o], y[o]
    e=0.0; rows=[]
    for b in np.array_split(np.arange(len(conf)), n_bins):
        if len(b)==0: continue
        c=conf[b].mean(); a=y[b].mean(); w=len(b)/len(conf)
        e += w*abs(a-c)
        rows.append(dict(n=len(b), mean_confidence=c, observed_accuracy=a, gap=a-c))
    return float(e), pd.DataFrame(rows)

def auroc(score, correct):
    s=np.asarray(score,float); y=np.asarray(correct,bool)
    n1=int(y.sum()); n0=int(len(y)-n1)
    if n1==0 or n0==0: return float("nan")
    r=pd.Series(s).rank().values
    return float((r[y].sum()-n1*(n1+1)/2)/(n1*n0))

def brier(conf, correct):
    return float(np.mean((np.asarray(conf,float)-np.asarray(correct,float))**2))

def wilson(k, n, alpha=None):
    """Wilson score interval for a proportion. Stable at the extremes, unlike normal approx."""
    alpha = alpha or CFG["ALPHA"]
    if n==0: return (float("nan"), float("nan"))
    from statistics import NormalDist
    z = NormalDist().inv_cdf(1-alpha/2)
    p = k/n; d = 1+z*z/n
    c = (p + z*z/(2*n))/d
    h = z*sqrt(p*(1-p)/n + z*z/(4*n*n))/d
    return (max(0.0,c-h), min(1.0,c+h))

def boot_ci(x, stat=np.mean, B=None, seed=None):
    x=np.asarray(x,float); B=B or CFG["N_BOOT"]
    rng=np.random.default_rng(seed or CFG["SEED_BOOT"])
    s=[stat(x[rng.integers(0,len(x),len(x))]) for _ in range(B)]
    return float(np.percentile(s,100*CFG["ALPHA"]/2)), float(np.percentile(s,100*(1-CFG["ALPHA"]/2)))

def mcnemar(a, b):
    a,b=np.asarray(a,bool),np.asarray(b,bool)
    bb,cc=int(np.sum(a&~b)),int(np.sum(~a&b)); n=bb+cc
    p=1.0 if n==0 else min(1.0, sum(comb(n,j) for j in range(bb,n+1))*0.5**n)
    return bb,cc,n,p
print("scoring and statistics defined")

## Step 5 — The perturbations

Identical to RQ2. Four patient-reported fields are degraded. The physician examination and the
documented absent findings are never touched, in any condition, for any model.

In [ ]:
LAY = {
 "dyspnea":"trouble breathing","dyspnoea":"trouble breathing","shortness of breath":"trouble breathing",
 "dysuria":"burning when passing urine","hematuria":"blood in the urine","haematuria":"blood in the urine",
 "melena":"black tarry stools","haematemesis":"vomiting blood","hematemesis":"vomiting blood",
 "dysphagia":"trouble swallowing","odynophagia":"painful swallowing","arthralgia":"joint pain",
 "myalgia":"muscle aches","myalgias":"muscle aches","cephalgia":"headache",
 "palpitations":"heart racing","palpitation":"heart racing","syncope":"fainting","presyncope":"nearly fainting",
 "erythema":"redness","erythematous":"red","pruritus":"itching","pruritic":"itchy",
 "jaundice":"yellow skin and eyes","icterus":"yellow eyes","pallor":"looking pale",
 "edema":"swelling","oedema":"swelling","diaphoresis":"sweating","diaphoretic":"sweaty",
 "emesis":"vomiting","nausea":"feeling sick","anorexia":"loss of appetite",
 "constipation":"not able to open bowels","diarrhea":"loose stools","diarrhoea":"loose stools",
 "tenderness":"sore to touch","tender":"sore to touch","abdominal":"tummy","abdomen":"tummy",
 "epigastric":"upper tummy","periumbilical":"around the belly button","umbilicus":"belly button",
 "suprapubic":"lower tummy","flank":"side of the back","lumbar":"lower back",
 "thoracic":"chest","sternum":"breastbone","substernal":"behind the breastbone",
 "dizziness":"feeling dizzy","vertigo":"the room spinning","tinnitus":"ringing in the ears",
 "otalgia":"earache","otorrhea":"discharge from the ear","rhinorrhea":"runny nose",
 "rhinorrhoea":"runny nose","nasal congestion":"blocked nose","congestion":"blocked nose",
 "pharyngitis":"sore throat","odontalgia":"toothache","epistaxis":"nosebleed",
 "hemoptysis":"coughing up blood","haemoptysis":"coughing up blood","sputum":"phlegm",
 "productive cough":"cough bringing up phlegm","wheeze":"whistling when breathing",
 "wheezing":"whistling when breathing","orthopnea":"breathless lying flat",
 "paroxysmal nocturnal dyspnea":"waking up breathless at night",
 "claudication":"leg pain when walking","paresthesia":"pins and needles",
 "paraesthesia":"pins and needles","hypoesthesia":"numbness","numbness":"no feeling",
 "weakness":"feeling weak","fatigue":"feeling tired all the time","malaise":"feeling unwell",
 "lethargy":"no energy","pyrexia":"fever","febrile":"feverish","afebrile":"no fever",
 "rigors":"shaking chills","chills":"feeling cold and shivery","night sweats":"sweating at night",
 "polyuria":"passing urine a lot","polydipsia":"feeling very thirsty","nocturia":"getting up at night to pass urine",
 "urinary frequency":"needing to pass urine often","urgency":"needing to rush to the toilet",
 "incontinence":"leaking urine","oliguria":"passing very little urine",
 "menorrhagia":"heavy periods","dysmenorrhea":"painful periods","amenorrhea":"periods stopped",
 "dyspareunia":"pain during sex","vaginal discharge":"discharge from the vagina",
 "lesion":"mark on the skin","lesions":"marks on the skin","macule":"flat spot","papule":"small bump",
 "vesicle":"small blister","pustule":"pus-filled spot","urticaria":"hives","rash":"rash",
 "lymphadenopathy":"swollen glands","hepatomegaly":"enlarged liver","splenomegaly":"enlarged spleen",
 "ascites":"fluid in the tummy","distension":"bloating","distention":"bloating","bloating":"feeling bloated",
 "exacerbated":"made worse","exacerbation":"flare-up","alleviated":"made better","radiating":"spreading",
 "radiates":"spreads","intermittent":"comes and goes","persistent":"keeps going","acute":"sudden",
 "chronic":"long-standing","bilateral":"on both sides","unilateral":"on one side",
 "proximal":"nearer the body","distal":"further from the body","extremities":"arms and legs",
 "lower extremities":"legs","upper extremities":"arms","visual disturbance":"trouble seeing",
 "blurred vision":"blurry sight","diplopia":"double vision","photophobia":"light hurting the eyes",
 "loss of consciousness":"blacking out","consciousness":"being aware","seizure":"a fit",
 "tremor":"shaking","stiffness":"feeling stiff","hypertension":"high blood pressure",
 "hypotension":"low blood pressure","tachycardia":"fast heartbeat","bradycardia":"slow heartbeat",
 "weight loss":"losing weight","weight gain":"putting on weight","dehydration":"being dried out",
 "trauma":"an injury","onset":"when it started","duration":"how long it lasted","severity":"how bad it is",
 "aggravating":"what makes it worse","relieving":"what makes it better","denies":"says no to",
 "complains of":"says they have","presents with":"comes in with","reports":"says",
}
VAGUE_MOD = {
 r"\b\d+\s*(day|days)\b":"a few days", r"\b\d+\s*(week|weeks)\b":"a few weeks",
 r"\b\d+\s*(month|months)\b":"a few months", r"\b\d+\s*(year|years)\b":"a long time",
 r"\b\d+\s*(hour|hours)\b":"a few hours", r"\bsevere\b":"bad", r"\bmoderate\b":"not too bad",
 r"\bmild\b":"a bit", r"\bsharp\b":"bad", r"\bdull\b":"odd", r"\bburning\b":"funny",
 r"\bthrobbing\b":"bad", r"\bconstant\b":"most of the time", r"\bintermittent\b":"on and off",
 r"\bright upper quadrant\b":"up on one side", r"\bleft upper quadrant\b":"up on one side",
 r"\bright lower quadrant\b":"down on one side", r"\bleft lower quadrant\b":"down on one side",
 r"\bepigastri\w*\b":"top of the tummy", r"\bperiumbilical\b":"middle of the tummy",
}
VAGUE_HIGH = dict(VAGUE_MOD); VAGUE_HIGH.update({
 r"\b\d+\b":"some", r"\bmorning\b":"sometimes", r"\bnight\b":"sometimes", r"\bevening\b":"sometimes",
 r"\bchest\b":"up top", r"\babdominal\b":"tummy", r"\babdomen\b":"tummy", r"\bback\b":"behind",
 r"\bradiat\w+\b":"goes somewhere else", r"\bworsening\b":"getting worse",
 r"\bgradual\w*\b":"slowly", r"\bsudden\w*\b":"all at once",
})
FRAG_DROP = set("""a an the is are was were be been being am do does did has have had
he she it they him her them his hers its their there here that this those these
of to in on at by for with from as and or but so then than when while which who whom
i you we us our your my me""".split())

PERTURBABLE = ["chief_complaints","presentation","physical_history","family_history"]
UNTOUCHED   = ["physical_examination","absent_findings"]
PLACEHOLDER = {"", "-", "--", "none", "nil", "n/a"}
FIELDS = ["chief_complaints","presentation","physical_examination",
          "physical_history","family_history","absent_findings"]

def is_blank(t): return str(t or "").strip().lower() in PLACEHOLDER
def split_findings(text):
    if is_blank(text): return []
    return [p.strip() for p in re.split(r"(?<=[.!?])\s+|\n+", str(text).strip()) if p.strip()]
def enumerate_findings(vig):
    return [(f,i,s) for f in PERTURBABLE for i,s in enumerate(split_findings(vig.get(f)))]
def drop_finding(vig, field, idx):
    out=dict(vig); segs=split_findings(vig.get(field))
    kept=[s for i,s in enumerate(segs) if i!=idx]
    out[field]=" ".join(kept) if kept else "-"
    return out
def build_text(v):
    head=f"{v.get('age')} {v.get('age_format','year')} old {v.get('sex','')}".strip()
    return "  ".join([head]+[str(v.get(f) or "").strip() for f in FIELDS]).strip()
def resolve_gold(c):
    dl=[norm(x) for x in c["gold_standard"]["differential_list"]]
    if not dl: return None,[]
    pr = dl[0] if CFG["GOLD_FIELD"]=="differential_list[0]" else norm(c["diagnosis"])
    return pr, [pr]+[x for x in dl if x!=pr]

def apply_removal(vig, n, rng):
    out=dict(vig)
    pool=[(f,i) for f in PERTURBABLE for i in range(len(split_findings(vig.get(f))))]
    if not pool: return out
    drop=set(rng.sample(pool, min(n,len(pool))))
    for f in PERTURBABLE:
        segs=split_findings(vig.get(f))
        if not segs: continue
        kept=[s for i,s in enumerate(segs) if (f,i) not in drop]
        out[f]=" ".join(kept) if kept else "-"
    return out

_LAY_RE = sorted(LAY.items(), key=lambda kv: -len(kv[0]))
def apply_reword(vig, pct, rng):
    out=dict(vig)
    for f in PERTURBABLE:
        t=str(vig.get(f) or "")
        if is_blank(t): continue
        for term,lay in _LAY_RE:
            t=re.sub(r"\b%s\b"%re.escape(term), lambda m: lay if rng.random()<pct else m.group(0), t, flags=re.I)
        out[f]=t
    return out

def apply_fragment(vig, share, rng):
    out=dict(vig)
    for f in PERTURBABLE:
        segs=split_findings(vig.get(f))
        if not segs: continue
        new=[]
        for s in segs:
            if rng.random()<share:
                new.append(" ".join(w for w in re.findall(r"[A-Za-z0-9'%/.-]+", s)
                                    if w.lower() not in FRAG_DROP).lower())
            else: new.append(s)
        out[f]=" ".join(new)
    return out

def apply_vague(vig, rules, rng):
    out=dict(vig)
    for f in PERTURBABLE:
        t=str(vig.get(f) or "")
        if is_blank(t): continue
        for pat,rep in rules.items(): t=re.sub(pat,rep,t,flags=re.I)
        out[f]=t
    return out

def perturb(vig, condition, case_number):
    rng=random.Random(f"{CFG['SEED_PERT']}|{case_number}|{condition}")
    if condition=="original":          return dict(vig)
    if condition.startswith("remove"): return apply_removal(vig,int(re.search(r"\d+",condition).group()),rng)
    if condition.startswith("reword"): return apply_reword(vig,int(re.search(r"\d+",condition).group())/100,rng)
    if condition=="fragment partial":  return apply_fragment(vig,0.5,rng)
    if condition=="fragment complete": return apply_fragment(vig,1.0,rng)
    if condition=="vague moderate":    return apply_vague(vig,VAGUE_MOD,rng)
    if condition=="vague high":        return apply_vague(vig,VAGUE_HIGH,rng)
    raise ValueError(condition)
print("perturbations defined")

## Step 6 — Build the case table, the candidate pool, and the difficulty labels

Difficulty comes from the three physicians in the benchmark, not from any model. It is used by
the abstention card and reported alongside it.

In [ ]:
rows=[]
for c in CASES:
    pr,gold = resolve_gold(c)
    if not gold: continue
    rows.append(dict(case_number=c["case_number"], primary=pr, gold=gold,
                     classification=c["classification"], vignette=c["vignette"]))
CASE_INFO = pd.DataFrame(rows)

DESC = {norm(k):v for k,v in DESCRIPTIONS.items()}
CANDIDATES = sorted({d for g in CASE_INFO.gold for d in g})
CAND_TEXT  = [f"{c}. {DESC[c]}" if DESC.get(c) else c for c in CANDIDATES]
CAND_IX    = {c:i for i,c in enumerate(CANDIDATES)}
_label_only = sum(1 for c in CANDIDATES if not DESC.get(c))
print(f"{len(CASE_INFO)} cases, {len(CANDIDATES)} candidates, "
      f"{len(CANDIDATES)-_label_only} with a sourced description, {_label_only} label-only")

# every condition's text, precomputed once and shared by all three models
TEXT = {}
for _,r in CASE_INFO.iterrows():
    for cond in CONDITIONS:
        TEXT[(r.case_number, cond)] = build_text(perturb(r.vignette, cond, r.case_number))
print(f"{len(TEXT)} texts built ({len(CASE_INFO)} cases x {len(CONDITIONS)} conditions)")

DOCS=["MA","TH","NJ"]
def _top(c, who, k=3): return [norm(x) for x in (c["predictions_normalized"].get(who) or [])][:k]
hard=[]
for c in CASES:
    pr,gold=resolve_gold(c)
    if not gold: continue
    dh=sum(1 for d in DOCS if pr in _top(c,d))
    hard.append(dict(case_number=c["case_number"], doctors_correct=dh, gold_len=len(gold),
                     difficulty={3:"easy (3/3)",2:"moderate (2/3)",1:"hard (1/3)",0:"very hard (0/3)"}[dh]))
HARD=pd.DataFrame(hard); HARD["is_hard"]=HARD.doctors_correct<=1
t=(HARD.groupby(["doctors_correct","difficulty"]).size().rename("cases").reset_index()
     .sort_values("doctors_correct",ascending=False))
t["share"]=(t.cases/len(HARD)).map("{:.1%}".format)
display(t)
print(f"physician reference: lowest physician M3 = {ACCURACY_MARK:.4f}  (the accuracy mark)")

## Step 7 — The three models

All three expose the same four methods, so nothing downstream knows which model produced a
number:

| Method | Returns |
|---|---|
| `fit(fold, texts, labels)` | the model, ready to score |
| `sims(texts)` | an *n* x 388 similarity matrix against every candidate |
| `sims_to(texts, j)` | similarity of each text to candidate *j* only, used by the attribution test |
| `release()` | frees the GPU before the next model loads |

`zeroshot` never trains, so it loads the encoder **once** and reuses it across all five folds.
`finetuned` has to reload the base encoder each fold, because training mutates it.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.model_selection import StratifiedKFold

def _free():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache(); torch.cuda.ipc_collect()
    except Exception:
        pass

class TfidfRetrieval:
    """Keyword search. Ranks the same 388 descriptions by TF-IDF cosine similarity."""
    name="tfidf_retrieval"; needs_gpu=False
    def fit(self, fold, texts, labels):
        self.vec = TfidfVectorizer(ngram_range=(1,2), min_df=1, sublinear_tf=True,
                                   stop_words="english").fit(list(texts)+CAND_TEXT)
        self.C = normalize(self.vec.transform(CAND_TEXT))      # 388 x V, sparse
        return self
    def sims(self, texts):
        return (normalize(self.vec.transform(list(texts))) @ self.C.T).toarray()
    def sims_to(self, texts, j):
        return np.asarray((normalize(self.vec.transform(list(texts))) @ self.C[j].T).todense()).ravel()
    def release(self):
        self.vec=None; self.C=None; _free()

class BiEncoder:
    """PubMedBERT bi-encoder. finetune=False is the zero-shot arm."""
    def __init__(self, finetune):
        self.finetune=finetune
        self.name="finetuned" if finetune else "zeroshot"
        self.needs_gpu=True
        self.m=None
    def _load_base(self):
        from sentence_transformers import SentenceTransformer
        m=SentenceTransformer(CFG["ENCODER"]); m.max_seq_length=CFG["MAX_SEQ_LEN"]
        return m
    def fit(self, fold, texts, labels):
        if self.finetune:
            from sentence_transformers import InputExample, losses
            from torch.utils.data import DataLoader
            import torch; torch.manual_seed(CFG["SEED_CV"])
            self.release()
            self.m=self._load_base()
            lut=dict(zip(CANDIDATES,CAND_TEXT))
            ex=[InputExample(texts=[q,lut[y]]) for q,y in zip(texts,labels) if y in lut]
            dl=DataLoader(ex, shuffle=True, batch_size=CFG["BATCH"], drop_last=True)
            self.m.fit(train_objectives=[(dl, losses.MultipleNegativesRankingLoss(self.m))],
                       epochs=CFG["EPOCHS"], warmup_steps=int(0.1*len(dl)*CFG["EPOCHS"]),
                       optimizer_params={"lr":CFG["LR"]}, show_progress_bar=False)
            del ex, dl; _free()
            self.C=self._encode(CAND_TEXT)
        else:
            if self.m is None:                 # load once, reuse for every fold
                self.m=self._load_base()
                self.C=self._encode(CAND_TEXT)
        return self
    def _encode(self, texts):
        import torch
        with torch.inference_mode():
            return self.m.encode(list(texts), normalize_embeddings=True,
                                 batch_size=CFG["ENCODE_BATCH"], show_progress_bar=False)
    def sims(self, texts):   return self._encode(texts) @ self.C.T
    def sims_to(self, texts, j): return self._encode(texts) @ self.C[j]
    def release(self):
        if self.m is not None:
            try: self.m.to("cpu")
            except Exception: pass
        self.m=None; self.C=None; _free()

def make_model(name):
    if name=="tfidf_retrieval": return TfidfRetrieval()
    if name=="zeroshot":        return BiEncoder(finetune=False)
    if name=="finetuned":       return BiEncoder(finetune=True)
    raise ValueError(name)
print("three models defined:", ", ".join(MODELS))

## Step 8 — Run one model

This runs a single model across five folds and writes **one predictions file for that model**.
A checkpoint is saved after each fold, and a model whose predictions file already exists is
skipped, so the three models can be run in three separate sessions on a tight GPU.

For each fold the function:

1. fits the model on the fold's training cases
2. fits the temperature on the fold's training cases only
3. scores every held-out case under every condition
4. runs the attribution test on the clean text, batched across the whole fold rather than one
   call per case

In [ ]:
def predictions_path(model_name, run_id=None):
    return os.path.join(CFG["OUTDIR"], f"predictions_{model_name}_{run_id or RUN_ID}.parquet")

def find_existing(model_name):
    hits=sorted(glob.glob(os.path.join(CFG["OUTDIR"], f"predictions_{model_name}_*.parquet")))
    hits+=sorted(glob.glob(os.path.join(CFG["OUTDIR"], f"predictions_{model_name}_*.csv")))
    return hits[-1] if hits else None

def _save(df, model_name):
    p=predictions_path(model_name)
    df=df.copy(); df["ranked"]=df["ranked"].apply(list)
    try:
        df.to_parquet(p, index=False)
    except Exception:
        p=p.replace(".parquet",".csv"); df.to_csv(p, index=False)
    return p

def run_model(model_name, force=False):
    found=find_existing(model_name)
    if found and not force:
        print(f"[{model_name}] found {os.path.basename(found)} - skipping. Pass force=True to rerun.")
        return pd.read_parquet(found) if found.endswith(".parquet") else pd.read_csv(found)

    print(f"\n=== {model_name} ===")
    arm=make_model(model_name)
    skf=StratifiedKFold(n_splits=CFG["N_FOLDS"], shuffle=True, random_state=CFG["SEED_CV"])
    recs=[]; taus={}
    for fold,(tr,te) in enumerate(skf.split(CASE_INFO, CASE_INFO.classification), start=1):
        ck=os.path.join(CFG["OUTDIR"], f"_ckpt_{model_name}_fold{fold}.json")
        if os.path.exists(ck) and not force:
            part=json.load(open(ck)); recs+=part["recs"]; taus[fold]=part["tau"]
            print(f"   fold {fold}: restored from checkpoint ({len(part['recs'])} records)")
            continue

        trd=CASE_INFO.iloc[tr]; ted=CASE_INFO.iloc[te]
        tr_text=[TEXT[(cn,"original")] for cn in trd.case_number]
        arm.fit(fold, tr_text, list(trd.primary))

        tau=fit_temperature(arm.sims(tr_text), [CAND_IX.get(p,-1) for p in trd.primary])
        taus[fold]=tau

        te_cn=list(ted.case_number)
        gold={r.case_number:r.gold for _,r in ted.iterrows()}
        prim={r.case_number:r.primary for _,r in ted.iterrows()}

        # ---- score every condition -------------------------------------------------
        per_cond={}
        for cond in CONDITIONS:
            S=arm.sims([TEXT[(cn,cond)] for cn in te_cn])
            order=np.argsort(-S,axis=1)
            per_cond[cond]=dict(S=S, order=order,
                                conf_raw=topk_conf(S,1.0), conf_cal=topk_conf(S,tau))

        # ---- attribution, batched across the fold ----------------------------------
        S0=per_cond["original"]["S"]; o0=per_cond["original"]["order"]
        pred_ix=[int(o0[i][0]) for i in range(len(te_cn))]
        finds={cn: enumerate_findings(CASE_INFO.set_index("case_number").loc[cn,"vignette"])
               for cn in te_cn}
        flat=[]; span={}
        for i,cn in enumerate(te_cn):
            s=len(flat); flat+= [s for _,_,s in finds[cn]]; span[cn]=(s,len(flat))
        attr={}
        if flat:
            rng=random.Random(f"{CFG['SEED_ATTR']}|{model_name}|fold{fold}")
            dele=[]; meta=[]
            for i,cn in enumerate(te_cn):
                f=finds[cn]
                if len(f)<2:
                    attr[cn]=dict(attr_field=None,attr_idx=None,attr_text=None,
                                  d_attr=np.nan,d_rand=np.nan,n_findings=len(f)); continue
                a0,a1=span[cn]
                sim=arm.sims_to(flat[a0:a1], pred_ix[i])
                a=int(np.argmax(sim)); b=rng.choice([j for j in range(len(f)) if j!=a])
                fa,ia,ta=f[a]; fb,ib,_=f[b]
                vig=CASE_INFO.set_index("case_number").loc[cn,"vignette"]
                meta.append((cn,i,fa,ia,ta))
                dele.append(build_text(drop_finding(vig,fa,ia)))
                dele.append(build_text(drop_finding(vig,fb,ib)))
            for k,(cn,i,fa,ia,ta) in enumerate(meta):
                base=float(S0[i,pred_ix[i]])
                sa=float(arm.sims_to([dele[2*k]],   pred_ix[i])[0])
                sb=float(arm.sims_to([dele[2*k+1]], pred_ix[i])[0])
                attr[cn]=dict(attr_field=fa, attr_idx=ia, attr_text=str(ta)[:120],
                              d_attr=base-sa, d_rand=base-sb, n_findings=len(finds[cn]))

        # ---- write records ---------------------------------------------------------
        fold_recs=[]
        for cond in CONDITIONS:
            pc=per_cond[cond]
            for i,cn in enumerate(te_cn):
                ranked=[CANDIDATES[j] for j in pc["order"][i][:CFG["K_RETRIEVE"]]]
                rec=dict(model=model_name, fold=fold, case_number=int(cn), condition=cond,
                         primary=prim[cn], predicted=ranked[0], ranked=ranked,
                         M1=hit_at_k(ranked,prim[cn],1), M3=hit_at_k(ranked,prim[cn],3),
                         M5=hit_at_k(ranked,prim[cn],5),
                         sim_top1=float(pc["S"][i,pc["order"][i][0]]),
                         conf_raw=float(pc["conf_raw"][i]), conf_cal=float(pc["conf_cal"][i]),
                         tau=float(tau))
                if cond=="original": rec.update(attr.get(cn, {}))
                fold_recs.append(rec)
        json.dump({"tau":float(tau),"recs":fold_recs}, open(ck,"w"), default=str)
        recs+=fold_recs
        print(f"   fold {fold}: tau {tau:.4f}; {len(ted)} cases x {len(CONDITIONS)} conditions")
        if arm.needs_gpu and arm.finetune: _free()

    arm.release(); del arm; _free()
    df=pd.DataFrame(recs).merge(HARD, on="case_number", how="left")
    p=_save(df, model_name)
    print(f"[{model_name}] {len(df)} records -> {os.path.basename(p)}")
    return df

Run the three. On a tight GPU, run this cell once per model by editing `TO_RUN`.

In [ ]:
TO_RUN = MODELS          # e.g. ["finetuned"] to run just one in this session

PRED = {}
for name in TO_RUN:
    PRED[name] = run_model(name)

for name in MODELS:
    if name not in PRED:
        f = find_existing(name)
        if f: PRED[name] = pd.read_parquet(f) if f.endswith(".parquet") else pd.read_csv(f)

missing=[m for m in MODELS if m not in PRED]
if missing: print("\nSTILL TO RUN:", ", ".join(missing))
else:
    display(pd.DataFrame([{"model":m,"records":len(PRED[m]),
                           "clean M3":round(PRED[m].query("condition=='original'").M3.mean(),4)}
                          for m in MODELS]))

## Step 9 — Grade one model on the five properties

Every number below is read from that model's predictions file. Pass or fail is decided from the
confidence interval, not the point estimate.

In [ ]:
def grade(df):
    """Grade one model. Returns a dict of five properties, each with value, CI, mark, verdict."""
    clean = df[df.condition=="original"].sort_values("case_number").reset_index(drop=True)
    n = len(clean)
    out = {}

    # ---- 1. accuracy ------------------------------------------------------------
    k = int(clean.M3.sum()); m3 = k/n
    lo,hi = wilson(k,n)
    out["accuracy"] = dict(value=m3, lo=lo, hi=hi,
                           mark=MARKS["accuracy"]["value"],
                           pass_=hi >= MARKS["accuracy"]["value"],
                           detail=f"M3 {m3:.4f}, CI [{lo:.4f}, {hi:.4f}]")

    # ---- 2. robustness ----------------------------------------------------------
    preserving=[c for c,t in REGISTERED.items() if t=="preserving" and c in set(df.condition)]
    base = clean.set_index("case_number").M3
    keep=[]
    for cond in preserving:
        p = df[df.condition==cond].set_index("case_number").M3
        idx = base[base].index.intersection(p.index)
        keep.append(pd.Series(p.loc[idx].values, index=idx, name=cond))
    if keep:
        R = pd.concat(keep, axis=1)
        held = int(R.all(axis=1).sum()); tot = len(R)       # right on clean AND right after every
        ret = held/tot if tot else float("nan")             # information-preserving perturbation
        rlo,rhi = wilson(held, tot)
    else:
        ret=rlo=rhi=float("nan"); held=tot=0
    out["robustness"] = dict(value=ret, lo=rlo, hi=rhi,
                             mark=MARKS["robustness"]["value"],
                             pass_=(rlo >= MARKS["robustness"]["value"]) if tot else False,
                             detail=f"{held} of {tot} correct answers survive all "
                                    f"{len(preserving)} preserving conditions")

    # ---- 3. calibration ---------------------------------------------------------
    ece, bins = ece_bins(clean.conf_cal, clean.M3)
    out["calibration"] = dict(value=ece, lo=np.nan, hi=np.nan,
                              mark=MARKS["calibration"]["value"],
                              pass_=ece <= MARKS["calibration"]["value"],
                              detail=f"ECE {ece:.4f}; AUROC {auroc(clean.conf_cal, clean.M3):.4f}; "
                                     f"Brier {brier(clean.conf_cal, clean.M3):.4f}",
                              bins=bins)

    # ---- 4. abstention ----------------------------------------------------------
    o = np.argsort(-clean.conf_cal.values)
    y = clean.M3.values.astype(bool)[o]
    acc = np.cumsum(y)/np.arange(1,n+1)
    ok = np.where(acc >= ACCURACY_MARK)[0]
    cov = (ok.max()+1)/n if len(ok) else 0.0       # largest coverage still meeting the accuracy mark
    clo,chi = wilson(int(round(cov*n)), n)
    out["abstention"] = dict(value=cov, lo=clo, hi=chi,
                             mark=MARKS["abstention"]["value"],
                             pass_=clo >= MARKS["abstention"]["value"],
                             detail=f"answers {cov:.1%} of cases at M3 >= {ACCURACY_MARK:.4f}")

    # ---- 5. auditability --------------------------------------------------------
    F = clean.dropna(subset=["d_attr","d_rand"])
    w = int((F.d_attr > F.d_rand).sum()); m = len(F)
    share = w/m if m else float("nan")
    alo,ahi = wilson(w,m)
    out["auditability"] = dict(value=share, lo=alo, hi=ahi,
                               mark=MARKS["auditability"]["value"],
                               pass_=(alo >= MARKS["auditability"]["value"]) if m else False,
                               detail=f"credited finding beats a random one in {w} of {m} cases")
    return out

GRADES = {m: grade(PRED[m]) for m in MODELS if m in PRED}
print("graded:", ", ".join(GRADES))

---
# The three-model table

One row per model. This is the main result.

In [ ]:
PROPS=["accuracy","robustness","calibration","abstention","auditability"]
rows=[]
for m in MODELS:
    if m not in GRADES: continue
    g=GRADES[m]; r={"model":m}
    for p in PROPS:
        r[p]=f"{g[p]['value']:.4f}"
        r[p+" "]=("PASS" if g[p]["pass_"] else "FAIL")
    npass=sum(g[p]["pass_"] for p in PROPS)
    r["properties passed"]=f"{npass} of 5"
    r["fit for use"]="YES" if npass==5 else "NO"
    rows.append(r)
MAIN=pd.DataFrame(rows)
display(MAIN)

detail=[]
for m in MODELS:
    if m not in GRADES: continue
    for p in PROPS:
        g=GRADES[m][p]
        ci = "" if np.isnan(g["lo"]) else f"[{g['lo']:.4f}, {g['hi']:.4f}]"
        detail.append({"model":m,"property":p,"value":round(g["value"],4),"95% CI":ci,
                       "mark":g["mark"],
                       "direction":"at least" if MARKS[p]["direction"]=="min" else "at most",
                       "verdict":"PASS" if g["pass_"] else "FAIL","detail":g["detail"]})
DETAIL=pd.DataFrame(detail)
display(DETAIL)

---
# The three hypotheses

Each is decided from the table above. Each can fail.

In [ ]:
present=[m for m in MODELS if m in GRADES]

# ---- H1: does the framework separate models that accuracy cannot separate? -------------
pairs=[]
for i in range(len(present)):
    for j in range(i+1,len(present)):
        a,b=present[i],present[j]; ga,gb=GRADES[a]["accuracy"],GRADES[b]["accuracy"]
        overlap = not (ga["hi"] < gb["lo"] or gb["hi"] < ga["lo"])
        sep=[]
        for p in PROPS[1:]:
            x,y=GRADES[a][p],GRADES[b][p]
            if np.isnan(x["lo"]) or np.isnan(y["lo"]):
                if abs(x["value"]-y["value"])>0: sep.append(f"{p} (no CI, differs)")
            elif x["hi"] < y["lo"] or y["hi"] < x["lo"]:
                sep.append(p)
        pairs.append({"model A":a,"model B":b,
                      "M3 A":round(ga["value"],4),"M3 B":round(gb["value"],4),
                      "accuracy intervals overlap":"yes" if overlap else "no",
                      "properties that separate them":", ".join(sep) if sep else "none",
                      "n separating":len(sep)})
P1=pd.DataFrame(pairs, columns=["model A","model B","M3 A","M3 B",
      "accuracy intervals overlap","properties that separate them","n separating"])
display(P1)
if len(present) < 2:
    print("H1 needs at least two models; run the others before judging it.")
    h1 = None
else:
    h1 = bool(((P1["accuracy intervals overlap"]=="yes") & (P1["n separating"]>0)).any())
if h1 is not None:
    print(f"H1 {'SUPPORTED' if h1 else 'NOT SUPPORTED'} - "
          f"{'at least one accuracy-tied pair is separated by another property' if h1 else 'no accuracy-tied pair is separated'}")

# ---- H2: does the five-property verdict change the decision accuracy alone would give? --
rows=[]
for m in present:
    g=GRADES[m]; acc=g["accuracy"]["pass_"]; others=[p for p in PROPS[1:] if not g[p]["pass_"]]
    rows.append({"model":m,
                 "accuracy-only verdict":"fit" if acc else "not fit",
                 "five-property verdict":"fit" if (acc and not others) else "not fit",
                 "fails":", ".join(others) if others else "none",
                 "verdicts differ": acc and bool(others)})
P2=pd.DataFrame(rows); display(P2)
h2 = bool(P2["verdicts differ"].any())
print(f"H2 {'SUPPORTED' if h2 else 'NOT SUPPORTED'} - "
      f"{'at least one model passes accuracy and fails another property' if h2 else 'every model meeting the accuracy mark meets all five'}")

# ---- H3: are the verdicts stable when each mark moves by half its interval? ------------
rows=[]
for m in present:
    g=GRADES[m]; base=sum(g[p]["pass_"] for p in PROPS)==5
    for p in PROPS:
        x=g[p]
        half = 0.02 if np.isnan(x["lo"]) else (x["hi"]-x["lo"])/2
        flips=[]
        for sign in (-1,+1):
            mk = x["mark"] + sign*half
            if MARKS[p]["direction"]=="min":
                ok = (x["hi"] if p=="accuracy" else x["lo"]) >= mk
            else:
                ok = x["value"] <= mk
            others=all(g[q]["pass_"] for q in PROPS if q!=p)
            if (ok and others) != base: flips.append(f"{sign:+.0f}")
        rows.append({"model":m,"property":p,"mark":x["mark"],
                     "moved by":round(half,4),
                     "verdict flips":", ".join(flips) if flips else "no"})
P3=pd.DataFrame(rows); display(P3[P3["verdict flips"]!="no"] if (P3["verdict flips"]!="no").any() else P3)
h3 = not (P3["verdict flips"]!="no").any()
print(f"H3 {'SUPPORTED' if h3 else 'NOT SUPPORTED'} - "
      f"{'no verdict flips' if h3 else 'at least one verdict flips when a mark moves by half its interval'}")

def _v(x): return "needs all three models" if x is None else ("SUPPORTED" if x else "NOT SUPPORTED")
VERDICTS=pd.DataFrame([
 ("H1","Does five-property evaluation separate accuracy-tied models?",_v(h1)),
 ("H2","Does the five-property verdict change the decision?",_v(h2)),
 ("H3","Are the verdicts stable when the marks move?",_v(h3)),
], columns=["hypothesis","question","verdict"])
display(VERDICTS)

## Step 10 — Save

One workbook with every table, plus the three predictions files already written in Step 8.

In [ ]:
xlsx=os.path.join(CFG["OUTDIR"], f"five_property_tables_{RUN_ID}.xlsx")
with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
    MAIN.to_excel(xw, sheet_name="three-model table", index=False)
    DETAIL.to_excel(xw, sheet_name="property detail", index=False)
    P1.to_excel(xw, sheet_name="H1 separation", index=False)
    P2.to_excel(xw, sheet_name="H2 verdicts", index=False)
    P3.to_excel(xw, sheet_name="H3 sensitivity", index=False)
    VERDICTS.to_excel(xw, sheet_name="hypotheses", index=False)
    pd.DataFrame([{"property":k,"mark":v["value"],
                   "direction":"at least" if v["direction"]=="min" else "at most",
                   "source":v["source"]} for k,v in MARKS.items()]
                 ).to_excel(xw, sheet_name="pass marks", index=False)
    for m in MODELS:
        if m in GRADES and "bins" in GRADES[m]["calibration"]:
            GRADES[m]["calibration"]["bins"].round(4).to_excel(
                xw, sheet_name=f"reliability {m[:14]}", index=False)

json.dump({"run_id":RUN_ID, "settings":{k:str(v) for k,v in CFG.items()},
           "marks":{k:{"value":v["value"],"direction":v["direction"],"source":v["source"]}
                    for k,v in MARKS.items()},
           "models":{m:{p:{"value":float(GRADES[m][p]["value"]),
                           "lo":None if np.isnan(GRADES[m][p]["lo"]) else float(GRADES[m][p]["lo"]),
                           "hi":None if np.isnan(GRADES[m][p]["hi"]) else float(GRADES[m][p]["hi"]),
                           "mark":GRADES[m][p]["mark"], "pass":bool(GRADES[m][p]["pass_"])}
                        for p in PROPS} for m in GRADES},
           "hypotheses":{"H1":(None if h1 is None else bool(h1)),"H2":bool(h2),"H3":bool(h3)}},
          open(os.path.join(CFG["OUTDIR"], f"five_property_summary_{RUN_ID}.json"),"w"), indent=1)

files=[{"file":os.path.basename(find_existing(m)),"contains":f"{len(PRED[m])} records, {m}"}
       for m in MODELS if m in PRED]
files+=[{"file":os.path.basename(xlsx),"contains":"every table"},
        {"file":f"five_property_summary_{RUN_ID}.json","contains":"headline numbers"}]
display(pd.DataFrame(files))
print("\nEvery table above was built from the three predictions files and nothing else.")